In [ ]:
import gdown
id = "1LfAv_C1biyjBNCjq8i5H_aAiKZX6Wy39"
output = "./datalake/"
gdown.download_folder(id=id, output=output)

In [55]:
import pandas as pd
import json
import os
import glob
import re

In [8]:
def load_json_files(directory_path):
    """
    Load all JSON files from a given directory
    """
    json_files = glob.glob(os.path.join(directory_path, '*.json')) + \
                 glob.glob(os.path.join(directory_path, '*.jsonl'))
    
    all_data = []
    for file in json_files:
        try:
            with open(file, 'r', encoding='utf-8') as f:
                # Handle both JSON and JSONL formats
                if file.endswith('.json'):
                    data = json.load(f)
                else:  # JSONL
                    data = [json.loads(line) for line in f]
                all_data.extend(data)
        except Exception as e:
            print(f"Error reading {file}: {e}")
    
    return all_data

def clean_and_process_books(book_data):
    """
    Clean and process book data
    """
    cleaned_books = []
    series_set = set()
    genre_set = set()
    
    for book in book_data:
        # Handle series as a list or string
        series = book.get('series', '')
        if isinstance(series, list):
            series = series[0] if series else ''
        
        # Clean and standardize book data
        cleaned_book = {
            'isbn': book.get('isbn', [''])[0] if isinstance(book.get('isbn'), list) else book.get('isbn', ''),
            'isbn13': book.get('isbn13', [''])[0] if isinstance(book.get('isbn13'), list) else book.get('isbn13', ''),
            'title': book.get('title', ''),
            'image_url': book.get('imageUrl', ''),
            'description': book.get('description', ''),
            'author': book.get('author', ''),
            'num_pages': book.get('numPages', None),
            'language': book.get('language', ''),
            'review_count': book.get('reviewsCount', 0),
            'kindle_price': book.get('ebookPrice', None),
            'publish_date': book.get('publishDate', ''),
            'crawl_source': book.get('crawlSource', ''),
            'average_rating': book.get('averageRating', None),
            'series': series
        }
        
        # Track series
        if cleaned_book['series']:
            series_set.add(cleaned_book['series'])
        
        # Track genres
        genres = book.get('genres', [])
        if isinstance(genres, list):
            genre_set.update(genres)
        elif isinstance(genres, str):
            genre_set.add(genres)
        
        cleaned_books.append(cleaned_book)
    
    return cleaned_books, series_set, genre_set

def process_reviews(review_data):
    """
    Process review data
    """
    reviews = []
    for review_set in review_data:
        # Handle different review formats
        if 'rating' in review_set:
            for rating_entry in review_set['rating']:
                reviews.append({
                    'isbn': review_set.get('isbn', [''])[0] if isinstance(review_set.get('isbn'), list) else review_set.get('isbn', ''),
                    'isbn13': review_set.get('isbn13', [''])[0] if isinstance(review_set.get('isbn13'), list) else review_set.get('isbn13', ''),
                    'rating_score': rating_entry.get('rating', 0) if isinstance(rating_entry, dict) else rating_entry
                })
    
    return reviews

def create_rating_distribution(reviews_df):
    """
    Create rating distribution for each book
    Handle variable number of rating scores
    """
    # Group by book and rating, count occurrences
    rating_dist = reviews_df.groupby(['isbn', 'isbn13', 'rating_score']).size().reset_index(name='count')
    
    # Pivot the table to create columns for each rating
    rating_dist_pivot = rating_dist.pivot_table(
        index=['isbn', 'isbn13'], 
        columns='rating_score', 
        values='count', 
        fill_value=0
    ).reset_index()
    
    # Rename columns to match expected format, handling missing ratings
    column_mapping = {
        0: 'zero_star_count',
        1: 'one_star_count',
        2: 'two_stars_count',
        3: 'three_stars_count',
        4: 'four_stars_count',
        5: 'five_stars_count'
    }
    
    # Rename columns, adding missing columns if they don't exist
    new_columns = {}
    for i in range(6):  # 0 to 5 stars
        col_name = column_mapping.get(i, f'{i}_star_count')
        new_columns[i] = rating_dist_pivot[i] if i in rating_dist_pivot.columns else 0
    
    # Create final DataFrame
    final_rating_dist = pd.DataFrame({
        'isbn': rating_dist_pivot['isbn'],
        'isbn13': rating_dist_pivot['isbn13'],
        'zero_star_count': new_columns[0],
        'one_star_count': new_columns[1],
        'two_stars_count': new_columns[2],
        'three_stars_count': new_columns[3],
        'four_stars_count': new_columns[4],
        'five_stars_count': new_columns[5]
    })
    
    return final_rating_dist

In [6]:
def main(directory_path):
    # Load data
    book_data = load_json_files(os.path.join(directory_path, 'goodreads'))
    
    # Process books
    cleaned_books, series_set, genre_set = clean_and_process_books(book_data)
    
    # Create DataFrames
    books_df = pd.DataFrame(cleaned_books)
    
    # Series DataFrame
    series_df = pd.DataFrame({
        'series_id': range(1, len(series_set) + 1),
        'name': list(series_set)
    })
    
    # Genres DataFrame
    genres_df = pd.DataFrame({
        'genre_id': range(1, len(genre_set) + 1),
        'name': list(genre_set)
    })
    
    # Book-Genres DataFrame
    book_genres = []
    for book in book_data:
        if book.get('genres'):
            for genre in book['genres']:
                book_genres.append({
                    'isbn': book.get('isbn', [''])[0] if isinstance(book.get('isbn'), list) else book.get('isbn', ''),
                    'isbn13': book.get('isbn13', [''])[0] if isinstance(book.get('isbn13'), list) else book.get('isbn13', ''),
                    'genre_id': genres_df[genres_df['name'] == genre]['genre_id'].values[0]
                })
    book_genres_df = pd.DataFrame(book_genres)
    
    # Process reviews
    review_data = load_json_files(os.path.join(directory_path, 'goodreads'))
    reviews = process_reviews(review_data)
    reviews_df = pd.DataFrame(reviews)
    
    # Rating distribution
    rating_dist_df = create_rating_distribution(reviews_df)
    
    # Save to CSV
    books_df.to_csv('books.csv', index=False)
    series_df.to_csv('series.csv', index=False)
    genres_df.to_csv('genres.csv', index=False)
    book_genres_df.to_csv('book_genres.csv', index=False)
    reviews_df.to_csv('reviews.csv', index=False)
    rating_dist_df.to_csv('rating_distribution.csv', index=False)
    
    print("CSV files created successfully!")

# Usage
# main('./datalake/book-rec-datalake')

In [9]:
main('./datalake/book-rec-datalake')

CSV files created successfully!


draft

In [10]:
import pandas as pd
import json
import os
import glob

def load_json_files(file_paths):
    """
    Load data from multiple JSON files
    
    Args:
    file_paths (list): List of file paths to load
    
    Returns:
    list: Combined data from all files
    """
    all_data = []
    for file_path in file_paths:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                # Handle both JSON and JSONL formats
                if file_path.endswith('.json'):
                    data = json.load(f)
                else:  # JSONL
                    data = [json.loads(line) for line in f]
                
                # Ensure data is a list
                if not isinstance(data, list):
                    data = [data]
                
                all_data.extend(data)
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
    
    return all_data




In [11]:
def process_books(book_data):
    """
    Process book data
    
    Args:
    book_data (list): List of book dictionaries
    
    Returns:
    tuple: Cleaned books, series set, genre set
    """
    cleaned_books = []
    series_set = set()
    genre_set = set()
    
    for book in book_data:
        # Handle series as a list or string
        series = book.get('series', '')
        if isinstance(series, list):
            series = series[0] if series else ''
        
        # Clean and standardize book data
        cleaned_book = {
            'isbn': book.get('isbn', [''])[0] if isinstance(book.get('isbn'), list) else book.get('isbn', ''),
            'isbn13': book.get('isbn13', [''])[0] if isinstance(book.get('isbn13'), list) else book.get('isbn13', ''),
            'title': book.get('title', ''),
            'image_url': book.get('imageUrl', ''),
            'description': book.get('description', ''),
            'author': book.get('author', ''),
            'num_pages': book.get('numPages', None),
            'language': book.get('language', ''),
            'review_count': book.get('reviewsCount', 0),
            'kindle_price': book.get('ebookPrice', None),
            'publish_date': book.get('publishDate', ''),
            'crawl_source': book.get('crawlSource', ''),
            'average_rating': book.get('averageRating', None),
            'series': series
        }
        
        # Track series
        if cleaned_book['series']:
            series_set.add(cleaned_book['series'])
        
        # Track genres
        genres = book.get('genres', [])
        if isinstance(genres, list):
            genre_set.update(genres)
        elif isinstance(genres, str):
            genre_set.add(genres)
        
        cleaned_books.append(cleaned_book)
    
    return cleaned_books, series_set, genre_set


In [12]:
def process_reviews(review_data):
    """
    Process review data
    
    Args:
    review_data (list): List of review dictionaries
    
    Returns:
    list: Processed reviews
    """
    reviews = []
    
    for review_set in review_data:
        # Handle reviews by book
        if 'rating' in review_set and isinstance(review_set['rating'], list):
            isbn = review_set.get('isbn', [''])[0] if isinstance(review_set.get('isbn'), list) else review_set.get('isbn', '')
            isbn13 = review_set.get('isbn13', [''])[0] if isinstance(review_set.get('isbn13'), list) else review_set.get('isbn13', '')
            
            for rating_entry in review_set['rating']:
                reviews.append({
                    'isbn': isbn,
                    'isbn13': isbn13,
                    'user_id': rating_entry.get('user', None),
                    'rating_score': rating_entry.get('rating', 0)
                })
        
        # Handle reviews by user
        elif 'userId' in review_set and 'rating' in review_set:
            user_id = review_set.get('userId', [''])[0] if isinstance(review_set.get('userId'), list) else review_set.get('userId', '')
            
            for rating_entry in review_set['rating']:
                reviews.append({
                    'isbn': rating_entry.get('isbn', ''),
                    'isbn13': rating_entry.get('isbn13', ''),
                    'user_id': user_id,
                    'rating_score': rating_entry.get('rating', 0)
                })
    
    return reviews



In [13]:
def process_users(user_data):
    """
    Process user data
    
    Args:
    user_data (list): List of user dictionaries
    
    Returns:
    list: Processed users
    """
    processed_users = []
    for user in user_data:
        # Handle variations in user data structure
        user_id = user.get('userId', [''])[0] if isinstance(user.get('userId'), list) else user.get('userId', '')
        name = user.get('name', [''])[0] if isinstance(user.get('name'), list) else user.get('name', '')
        
        processed_user = {
            'user_id': user_id,
            'name': name,
            'profile_url': user.get('profileUrl', [''])[0] if isinstance(user.get('profileUrl'), list) else user.get('profileUrl', ''),
            'total_ratings': user.get('totalRatings', ['0'])[0] if isinstance(user.get('totalRatings'), list) else user.get('totalRatings', '0'),
            'total_reviews': user.get('totalReviews', ['0'])[0] if isinstance(user.get('totalReviews'), list) else user.get('totalReviews', '0'),
            'avg_rating': user.get('avgRating', ['0'])[0] if isinstance(user.get('avgRating'), list) else user.get('avgRating', '0'),
            'image_url': user.get('imageUrl', [''])[0] if isinstance(user.get('imageUrl'), list) else user.get('imageUrl', '')
        }
        
        processed_users.append(processed_user)
    
    return processed_users



In [14]:
def create_rating_distribution(reviews_df):
    """
    Create rating distribution for each book
    
    Args:
    reviews_df (pd.DataFrame): DataFrame of reviews
    
    Returns:
    pd.DataFrame: Rating distribution
    """
    # Group by book and rating, count occurrences
    rating_dist = reviews_df.groupby(['isbn', 'isbn13', 'rating_score']).size().reset_index(name='count')
    
    # Pivot the table to create columns for each rating
    rating_dist_pivot = rating_dist.pivot_table(
        index=['isbn', 'isbn13'], 
        columns='rating_score', 
        values='count', 
        fill_value=0
    ).reset_index()
    
    # Rename columns to match expected format, handling missing ratings
    column_mapping = {
        0: 'zero_star_count',
        1: 'one_star_count',
        2: 'two_stars_count',
        3: 'three_stars_count',
        4: 'four_stars_count',
        5: 'five_stars_count'
    }
    
    # Prepare columns
    final_columns = {
        'isbn': rating_dist_pivot['isbn'],
        'isbn13': rating_dist_pivot['isbn13']
    }
    
    # Add star count columns
    for i in range(6):  # 0 to 5 stars
        col_name = column_mapping.get(i, f'{i}_star_count')
        final_columns[col_name] = rating_dist_pivot[i] if i in rating_dist_pivot.columns else 0
    
    # Create final DataFrame
    final_rating_dist = pd.DataFrame(final_columns)
    
    return final_rating_dist



In [71]:
def main(base_directory):
    # Define file paths
    books_json_files = [
        os.path.join(base_directory, 'goodreads', 'data_50000_60000.json'),
        os.path.join(base_directory, 'goodreads', 'data_60001_70000.json'),
        os.path.join(base_directory, 'goodreads', 'data_70001_80000.json'),
        os.path.join(base_directory, 'goodreads', 'data_80001_90000.json'),
        os.path.join(base_directory, 'goodreads/old', 'book_1_10k.json'),
        os.path.join(base_directory, 'goodreads/old', 'books_200000_to_250000.json'),
        # Add all book JSON file paths here
    ]
    
    reviews_json_files = [
        os.path.join(base_directory, 'goodreads', 'review_50000_60000.json'),
        # os.path.join(base_directory, 'goodreads', 'ratings.json'),
        os.path.join(base_directory, 'goodreads/old', 'reviews.jsonl'),
        os.path.join(base_directory, 'goodreads/old', 'reviews_books_200000_to_250000.json'),
        os.path.join(base_directory, 'goodreads/old', 'data_review1_10k.json'),
        os.path.join(base_directory, 'goodreads/old', 'data_review150-155k.json'),
        os.path.join(base_directory, 'goodreads/old', 'data_review155-160k.json'),
        os.path.join(base_directory, 'goodreads/old', 'data_review160-165k.json'),
        os.path.join(base_directory, 'goodreads/old', 'data_review165-170k.json'),
        os.path.join(base_directory, 'goodreads/old', 'data_review170-175k.json'),
        # Add all review JSON file paths here
    ]
    
    users_json_files = [
        os.path.join(base_directory, 'goodreads', 'group_user.json'),
        os.path.join(base_directory, 'goodreads', 'group1.json'),
        os.path.join(base_directory, 'goodreads/old', 'users.jsonl'),
        os.path.join(base_directory, 'goodreads', 'group_user.json'),
        # Add all user JSON file paths here
    ]
    
    # Load and process data
    print("Loading book data...")
    book_data = load_json_files(books_json_files)
    print(f"Loaded {len(book_data)} book entries")
    
    print("Processing book data...")
    cleaned_books, series_set, genre_set = process_books(book_data)
    
    print("Loading review data...")
    review_data = load_json_files(reviews_json_files)
    print(f"Loaded {len(review_data)} review entries")
    
    print("Processing review data...")
    reviews = process_reviews(review_data)
    
    print("Loading user data...")
    user_data = load_json_files(users_json_files)
    print(f"Loaded {len(user_data)} user entries")
    
    print("Processing user data...")
    users = process_users(user_data)
    
    # Create DataFrames
    books_df = pd.DataFrame(cleaned_books)
    reviews_df = pd.DataFrame(reviews)
    users_df = pd.DataFrame(users)
    
    # Series DataFrame
    series_df = pd.DataFrame({
        'series_id': range(1, len(series_set) + 1),
        'name': list(series_set)
    })
    
    # Genres DataFrame
    genres_df = pd.DataFrame({
        'genre_id': range(1, len(genre_set) + 1),
        'name': list(genre_set)
    })
    
    # Book-Genres DataFrame
    book_genres = []
    for book in book_data:
        if book.get('genres'):
            for genre in book.get('genres', []):
                book_genres.append({
                    'isbn': book.get('isbn', [''])[0] if isinstance(book.get('isbn'), list) else book.get('isbn', ''),
                    'isbn13': book.get('isbn13', [''])[0] if isinstance(book.get('isbn13'), list) else book.get('isbn13', ''),
                    'genre_id': genres_df[genres_df['name'] == genre]['genre_id'].values[0]
                })
    book_genres_df = pd.DataFrame(book_genres)

    
    # Rating distribution
    rating_dist_df = create_rating_distribution(reviews_df)
    
    # Save to CSV
    books_df.to_csv('books1.csv', index=False, mode='w')
    series_df.to_csv('series1.csv', index=False, mode='w')
    genres_df.to_csv('genres1.csv', index=False, mode='w')
    book_genres_df.to_csv('book_genres1.csv', index=False, mode='w')
    reviews_df.to_csv('reviews1.csv', index=False, mode='w')
    users_df.to_csv('users1.csv', index=False, mode='w')
    rating_dist_df.to_csv('rating_distribution1.csv', index=False, mode='w')
    
    print("CSV files created successfully!")


In [ ]:
main('./datalake/book-rec-datalake')

Loading book data...
Loaded 86343 book entries
Processing book data...
Loading review data...
Loaded 52237 review entries
Processing review data...
Loading user data...
Loaded 36670 user entries
Processing user data...


In [61]:
# with open('./datalake/book-rec-datalake/goodreads/ratings.json', 'r') as f:
#     data = f.read()
# print(data)

df = pd.read_json('./datalake/book-rec-datalake/goodreads/ratings.json')
df

,userId,rating
0,[145704009],"[{'isbn': '1400341124', 'isbn13': '97814003411..."
1,[8829118],"[{'isbn': '0812998626', 'isbn13': '97808129986..."
2,[63113280],"[{'isbn': '', 'isbn13': '9781984829993', 'book..."
3,[160270339],"[{'isbn': '0312642717', 'isbn13': '97803126427..."
4,[103379996],"[{'isbn': '', 'isbn13': '', 'book_href': 'http..."
...,...,...
1574,[107912421],"[{'isbn': '', 'isbn13': '', 'book_href': 'http..."
1575,[167510184],"[{'isbn': '', 'isbn13': '', 'book_href': 'http..."
1576,[92390881],"[{'isbn': '0525539727', 'isbn13': '97805255397..."
1577,[63665211],"[{'isbn': '0063308312', 'isbn13': '97800633083..."


In [62]:
exploded_df = df.explode('userId').reset_index(drop=True)
exploded_df = exploded_df.explode('rating').reset_index(drop=True)
exploded_df

,userId,rating
0,145704009,"{'isbn': '1400341124', 'isbn13': '978140034112..."
1,145704009,"{'isbn': '1662527004', 'isbn13': '978166252700..."
2,145704009,"{'isbn': '1662529376', 'isbn13': '978166252937..."
3,145704009,"{'isbn': '1662529341', 'isbn13': '978166252934..."
4,145704009,"{'isbn': '166252823X', 'isbn13': '978166252823..."
...,...,...
525779,125982174,"{'isbn': '0439554934', 'isbn13': '978043955493..."
525780,125982174,"{'isbn': '0316015849', 'isbn13': '978031601584..."
525781,125982174,"{'isbn': '0062024035', 'isbn13': '978006202403..."
525782,125982174,"{'isbn': '', 'isbn13': '', 'book_href': 'https..."


In [64]:
expanded_df = pd.concat(
    [exploded_df.drop(columns='rating'), pd.json_normalize(exploded_df['rating'])],
    axis=1
)
expanded_df

,userId,isbn,isbn13,book_href,rating
0,145704009,1400341124,9781400341122,https://www.goodreads.com/book/show/200212922-...,5
1,145704009,1662527004,9781662527005,https://www.goodreads.com/book/show/219754771-...,4
2,145704009,1662529376,9781662529375,https://www.goodreads.com/book/show/220389090-...,4
3,145704009,1662529341,9781662529344,https://www.goodreads.com/book/show/220389457-...,4
4,145704009,166252823X,9781662528231,https://www.goodreads.com/book/show/220389455-...,3
...,...,...,...,...,...
525779,125982174,0439554934,9780439554930,https://www.goodreads.com/book/show/3.Harry_Po...,5
525780,125982174,0316015849,9780316015844,https://www.goodreads.com/book/show/41865.Twil...,5
525781,125982174,0062024035,9780062024039,https://www.goodreads.com/book/show/13335037-d...,5
525782,125982174,,,https://www.goodreads.com/book/show/11870085-t...,5


In [65]:
print(f"Unique ISBN_10: {expanded_df['isbn'].nunique()}, ISBN_13: {expanded_df['isbn13'].nunique()}")
print(f"Unique userId: {expanded_df['userId'].nunique()}")

Unique ISBN_10: 141192, ISBN_13: 143252
Unique userId: 1579


In [69]:
df_to_save = expanded_df.rename(columns={
    "userId": "userid",
    "rating": "rating_score"
})

In [70]:
df_to_save.to_csv(
    'reviews1.csv',
    mode='a',
    index=False,
    header=not pd.io.common.file_exists('reviews2.csv'),
    columns=["isbn", "isbn13", "userid", "rating_score"]
)